# Lab 3
We'll use this lab as an experiment of using a single file where you fill in codeblocks where necessary. They will be available as .py and .ipynb. Using the latter, or Jupyter Notebook, is highly recommended, as it provides substantially better feedback.


Provide your outputs in a simple report, along with textual answers.


The idea behind this format is to clarify what sort of output is required, as all answers run on tests based in the `tests.py` file.

In [2]:
import sklearn
import nltk
import random
import pandas as pd
import re
# feel free to import from modules of sklearn and nltk later
# e.g., from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split

## Exercise 1 - Gender detection of names
In NLTK you'll find the corpus `corpus.names`. A set of 5000 male and 3000 female names.
1) Select a ratio of train/test data (based on experiences from previous labs perhaps?)
2) Build a feature extractor function
3) Build two classifiers:
    - Decision tree
    - Naïve bayes
    
Finally, write code to evaluate the classifiers. Explain your results, and what do you think would change if you altered your feature extractor?

In [143]:
class GenderDataset:
    def __init__(self):
        self.names = nltk.corpus.names
        self.data = None
        self.build()

    def make_labels(self, gender):
        """
        this function is to help you get started
        based on the passed gender, as you can fetch from the file ids,
        we return a tuple of (name, gender) for each name
        
        use this in `build` below, or do your own thing completely :)
        """
        return [(n, gender) for n in self.names.words(gender + ".txt")]
    
    def build(self):
        """ TODO
        combine the data in "male" and "female" into one
        remember to randomize the order
        """
        maleNames = self.make_labels("male")
        femaleNames = self.make_labels("female")
        self.data = maleNames + femaleNames
    
    def split(self, ratio):
        # I randomized the order with the shuffle attribute here
        return train_test_split(self.data, test_size=ratio, shuffle=True)

In [144]:
class Classifier:
    def __init__(self, classifier):
        self.classifier = classifier
        self.model = None
    
    def train(self, data):
        self.model = self.classifier.train(data)
        
    def test(self, data):
        return nltk.classify.accuracy(self.model, data)
    
    def train_and_evaluate(self, train, test):
        self.train(train)
        return self.test(test)
        
    def show_features(self):
        self.model.show_most_informative_features(5)

                                 
class FeatureExtractor:
    def __init__(self, data):
        self.data = data
        self.features = []
        
        self.build()
                 
    @staticmethod
    def text_to_features(name):
        # TODO: create a dict of features from a name
        return {
            "name": name,
            "lastLetter": name[-1],
            "2lastLetters": name[-2:]
        }
    
    def build(self):
       self.features = [(self.text_to_features(n), g) for (n,g) in self.data]


Note: you should achieve an accuracy of well above 70%!

In [145]:
split_ratio = 0.3  # Made the ratio 30% test and 70% train as we need a lot of data to train, but still be able to test on a large set of untouched data
train, test = GenderDataset().split(ratio=split_ratio)

classifiers = {
    "decision_tree": Classifier(nltk.DecisionTreeClassifier), 
    "naive_bayes": Classifier(nltk.NaiveBayesClassifier), 
}

train_set = FeatureExtractor(train).features
test_set = FeatureExtractor(test).features

for name, classifier in classifiers.items():
    acc = classifier.train_and_evaluate(train_set, test_set)
    print("Model: {}\tAccuracy: {}".format(name, acc))

classifiers["naive_bayes"].show_features()

Model: decision_tree	Accuracy: 0.6023489932885906
Model: naive_bayes	Accuracy: 0.7692953020134228
Most Informative Features
            2lastLetters = 'na'           female : male   =     72.1 : 1.0
            2lastLetters = 'ia'           female : male   =     68.9 : 1.0
            2lastLetters = 'la'           female : male   =     57.5 : 1.0
            2lastLetters = 'rt'             male : female =     39.9 : 1.0
              lastLetter = 'a'            female : male   =     39.0 : 1.0


I first started with only having the last character of the name and the name in the features. This did not give me the results I really wanted, probably because it does not really say much about the name. But then I experimented with using two letters from both the start and end of the name. This gave me a lot better results, as it might tell a little more about the name. But I saw that the first two letters didnt really change the accuracy much, so I decided to remove it to not overfit. The decision tree does not perform aswell, it might have performed better if the training sample was bigger. 

OUTPUT

```
Model: decision_tree	Accuracy: 0.5973154362416108
Model: naive_bayes	Accuracy: 0.7655201342281879
Most Informative Features
            2lastLetters = 'na'           female : male   =    123.5 : 1.0
            2lastLetters = 'la'           female : male   =     56.6 : 1.0
              lastLetter = 'k'              male : female =     56.2 : 1.0
            2lastLetters = 'ta'           female : male   =     55.8 : 1.0
            2lastLetters = 'rd'             male : female =     53.8 : 1.0
```


## Exercise 2 - Spam or ham
Spam or ham is referred to a mail being spam or regular ("ham"). Follow the instructions and implement the `TODOs`

In [146]:
spam = pd.read_csv(
    'spam.csv',
    usecols=["v1", "v2"],
    encoding="latin-1"
).rename(columns={"v1": "label", "v2": "text"})

print(spam.label.value_counts())
spam.head()

ham     4825
spam     747
Name: label, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [218]:
""" TODO: transform label to numerical
Expected output:
0    4825
1     747
Name: label, dtype: int64

hint: you can use "apply" or "replace" for a column in pandas
"""
spam["label"] = spam["label"].replace(["spam"], "1")
spam["label"] = spam["label"].replace(["ham"], "0")
spam.label.value_counts()

0    4825
1     747
Name: label, dtype: int64

OUTPUT
````
0    4825
1     747
Name: label, dtype: int64
````


In [23]:
class TextCleaner:
    def __init__(self, text):
        self.text = nltk.word_tokenize(text)
        self.stemmer = [nltk.stem.SnowballStemmer("english").stem(w) for w in self.text]
        self.stopwords = nltk.corpus.stopwords.words
        self.lem = [nltk.stem.WordNetLemmatizer.lemmatize(word=w, self=self) for w in self.text]

    """
    Create small functions to replace your tokens (self.text)
    iteratively. Such as a lowercase function.
    """
    def lowercase(self):
        self.text = [w.lower() for w in self.text]

    def stopwords(self):
        self.text = [w for w in self.text if w not in self.stopwords]

    def whitespaces(self):
        self.text = [w.strip() for w in self.text]

    def clean(self):
        self.lowercase()
        self.stopwords()
        self.whitespaces()
        """
        TODO: populate with your defined cleaning functions here
        perhaps you want some conditional values to
        control which functions to use?
        """
        
        # finally, return it as a text 
        return " ".join(self.text)

In [20]:
clean = lambda text: TextCleaner(text).clean()
spam.text = spam.text.apply(clean)

NameError: name 'spam' is not defined

In [221]:
spam.head()

,label,text
0,0,"go until jurong point , crazy .. available onl..."
1,0,ok lar ... joking wif u oni ...
2,1,free entry in 2 a wkly comp to win fa cup fina...
3,0,u dun say so early hor ... u c already then sa...
4,0,"nah i do n't think he goes to usf , he lives a..."


OUTPUT
```

label	text
0	ham	Go until jurong point, crazy.. Available only ...
1	ham	Ok lar... Joking wif u oni...
2	spam	Free entry in 2 a wkly comp to win FA Cup fina...
3	ham	U dun say so early hor... U c already then say...
4	ham	Nah I don't think he goes to usf, he lives aro...

```

In [227]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder

split_ratio = 0.3
X_train, X_test, y_train, y_test = train_test_split(
    spam.text, spam.label, test_size=split_ratio, random_state=4310)

# TODO: vectorize with sklearn
vectorizer = OneHotEncoder(handle_unknown='ignore')
# TODO: fit the vectorizer to your training data
X_train = vectorizer.fit_transform([[t, len(t)] for t in X_train])

# TODO: set up a multinomial classifier
classifier = MultinomialNB()
if classifier:
    classifier.fit(X_train, y_train)
    
vectorized = None

In [230]:
def predict(model, vectorizer, data, all_predictions=False):
    data = vectorizer.transform([[t, len(t)] for t in data])
    if all_predictions:
        return model.predict_proba(data)
    else:
        return model.predict(data)

def print_examples(data, probs, label1, label2, n=10):
    percent = lambda x: "{}%".format(round(x*100, 1))

    for text, pred in list(zip(data, probs))[:n]:
        print("{}\n{}: {} / {}: {}\n{}".format(
            text,
            label1,
            percent(pred[0]),
            label2,
            percent(pred[1]),
            "-" * 100  # to print a line
        ))

In [232]:
if classifier:
    y_probas = predict(classifier, vectorizer, X_test, all_predictions=True)
    print_examples(X_test, y_probas, "ham", "spam", 10)

    y_pred = predict(classifier, vectorizer, X_test)
    # TODO display a confusion matrix on the test set vs predictions
    confusion_mat = confusion_matrix(y_test, y_pred)
    print(confusion_mat)

    # show precision and recall in a confusion matrix
    tn, fp, fn, tp = confusion_mat.ravel()
    recall = tp / (tp + fn)
    precision = tp / (tp + fp)

    print("Recall={}\nPrecision={}".format(round(recall, 2), round(precision, 2)))

he is world famamus ....
ham: 99.6% / spam: 0.4%
----------------------------------------------------------------------------------------------------
\aww you must be nearly dead ! well jez iscoming over todo some workand that whilltake forever ! \ `` ``
ham: 90.0% / spam: 10.0%
----------------------------------------------------------------------------------------------------
how are you babes . hope your doing ok . i had a shit nights sleep . i fell asleep at 5.iåõm knackered and iåõm dreading work tonight . what are thou upto tonight . x
ham: 72.8% / spam: 27.2%
----------------------------------------------------------------------------------------------------
thank you . i like you as well ...
ham: 98.2% / spam: 1.8%
----------------------------------------------------------------------------------------------------
i 'm reading the text i just sent you . its meant to be a joke . so read it in that light
ham: 97.1% / spam: 2.9%
----------------------------------------------------

OUTPUT 
```
he is world famamus ....
ham: 99.6% / spam: 0.4%
----------------------------------------------------------------------------------------------------
\aww you must be nearly dead ! well jez iscoming over todo some workand that whilltake forever ! \ `` ``
ham: 90.0% / spam: 10.0%
----------------------------------------------------------------------------------------------------
how are you babes . hope your doing ok . i had a shit nights sleep . i fell asleep at 5.iåõm knackered and iåõm dreading work tonight . what are thou upto tonight . x
ham: 72.8% / spam: 27.2%
----------------------------------------------------------------------------------------------------
thank you . i like you as well ...
ham: 98.2% / spam: 1.8%
----------------------------------------------------------------------------------------------------
i 'm reading the text i just sent you . its meant to be a joke . so read it in that light
ham: 97.1% / spam: 2.9%
----------------------------------------------------------------------------------------------------
okie ì_ wan meet at bishan ? cos me at bishan now . i 'm not driving today .
ham: 96.8% / spam: 3.2%
----------------------------------------------------------------------------------------------------
smile in pleasure smile in pain smile when trouble pours like rain smile when sum1 hurts u smile becoz someone still loves to see u smiling ! !
ham: 73.7% / spam: 26.3%
----------------------------------------------------------------------------------------------------
hi : ) cts employee how are you ?
ham: 98.9% / spam: 1.1%
----------------------------------------------------------------------------------------------------
c movie is juz last minute decision mah . juz watch 2 lar but i tot ì_ not interested .
----------------------------------------------------------------------------------------------------
[[1449    0]
 [ 192   31]]
Recall=0.14
Precision=1.0
````

## Exercise 3 - Word features
Word features can be very useful for performing document classification, since the words that appear in a document give a strong indication of what its semantic content is. However, many words occur very infrequently, and some of the most informative words in a document may never have occurred in our training data. One solution is to make use of a lexicon, which describes how different words relate to each other.

Your task:
- Use the WordNet lexicon and augment the movie review document classifier (See NLTK book, Ch. 6, section 1.3) to use features that generalize the words that appear in a document, making it more likely that they will match words found in the training data.

Download wordnet and import

In [233]:
nltk.download('wordnet')
nltk.download("movie_reviews")
from nltk.corpus import movie_reviews
from nltk.corpus import wordnet as wn
import random

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jonathanbrooks/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/jonathanbrooks/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [319]:
# TODO: implement 
def word_to_syn(word) -> list:
    synsets = wn.synsets(word)
    syns = []

    if synsets == 0:
        return [word]

    for ss in synsets:
        syns += ss.lemma_names()

    return list(set(syns))

In [331]:
"""
this is from Ch. 6, sec. 1.3, with slight modifications
note that word_to_syn(word) (from the above implementation)
is in the beginning of the following function
"""
documents = [([word_to_syn(word) for word in list(movie_reviews.words(fileid))], category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

random.shuffle(documents)

all_words = nltk.FreqDist(w.lower() for w in movie_reviews.words())
n_most_freq = 2000
word_features = list(all_words)[:n_most_freq]

def document_features(document):
    document_words = set(document)
    features = {}
    for word in word_features:
        features['contains({})'.format(word)] = (word in document_words)
    return features

In [38]:
# Had to flatten the document, as I got an error when I tried to send in just d
featuresets = [(document_features([item for sublist in d for item in sublist]), c) for (d, c) in documents]

split_ratio = 0.4 # TODO: modify
train_set, test_set = train_test_split(featuresets, test_size=split_ratio)

# TODO: select a suitable classifier
classifier = nltk.NaiveBayesClassifier
model = classifier.train(train_set)

NameError: name 'documents' is not defined

In [333]:
# TODO: return a flattened list of input words and their lemmas
def synset_expansion(words) -> list:
    wnl = nltk.stem.WordNetLemmatizer()

    lemmas = []
    for w in words:
        syns = word_to_syn(w)
        lemmas += [wnl.lemmatize(syn) for syn in syns]
    
    return [lem.lower() for lem in lemmas]

expanded_word_features = synset_expansion(word_features)

In [334]:
# some assertions to test your code :-)
assert sorted(synset_expansion(["pc"])) == ["microcomputer", "pc", "personal_computer"]
assert sorted(synset_expansion(["programming", "coder"])) == [
    'coder',
    'computer_programing',
    'computer_programmer',
    'computer_programming',
    'program',
    'programing',
    'programme',
    'programmer',
    'programming',
    'scheduling',
    'software_engineer'
]

In [336]:
doc_featuresets = [(document_features([item for sublist in d for item in sublist]), c) for (d, c) in documents]
doc_train_set, doc_test_set = train_test_split(doc_featuresets, test_size=0.1)

doc_model = model.train(doc_train_set)
doc_model.show_most_informative_features(5)
print("Accuracy: ", nltk.classify.accuracy(doc_model, doc_test_set))

Most Informative Features
   contains(wonderfully) = True              pos : neg    =      4.7 : 1.0
       contains(unfunny) = True              neg : pos    =      4.4 : 1.0
           contains(era) = True              pos : neg    =      4.4 : 1.0
         contains(worst) = True              neg : pos    =      4.1 : 1.0
     contains(pointless) = True              neg : pos    =      3.7 : 1.0
Accuracy:  0.715


OUTPUT
```
Most Informative Features
   contains(wonderfully) = True              pos : neg    =      4.7 : 1.0
       contains(unfunny) = True              neg : pos    =      4.4 : 1.0
           contains(era) = True              pos : neg    =      4.4 : 1.0
         contains(worst) = True              neg : pos    =      4.1 : 1.0
     contains(pointless) = True              neg : pos    =      3.7 : 1.0
Accuracy:  0.715
```


In [337]:
def lexicon_features(reviews):
    review_words = set(reviews)
    features = {}
    for word in expanded_word_features:
        if word not in word_features:
            features['synset({})'.format(word)] = (word in review_words)
        features['contains({})'.format(word)] = (word in review_words)

    return features

Question: do you see any issues with including the synsets? Experiment a bit with different words and verify your ideas.

In [339]:
# warning: this may take some time to run
lex_featuresets = [(lexicon_features([item for sublist in d for item in sublist]), c) for (d, c) in documents]
lex_train_set, lex_test_set = train_test_split(lex_featuresets, test_size=0.1)
lex_model = model.train(lex_train_set)  # the same classifier as you defined above
lex_model.show_most_informative_features()
print("Accuracy: ", nltk.classify.accuracy(lex_model, lex_test_set))

Most Informative Features
         contains(owing) = True              pos : neg    =      9.7 : 1.0
  contains(undischarged) = True              pos : neg    =      9.7 : 1.0
           synset(owing) = True              pos : neg    =      9.7 : 1.0
    synset(undischarged) = True              pos : neg    =      9.7 : 1.0
    contains(squandered) = True              neg : pos    =      6.4 : 1.0
      synset(squandered) = True              neg : pos    =      6.4 : 1.0
      contains(farcical) = True              neg : pos    =      5.8 : 1.0
        synset(farcical) = True              neg : pos    =      5.8 : 1.0
     contains(atrophied) = True              neg : pos    =      5.6 : 1.0
       synset(atrophied) = True              neg : pos    =      5.6 : 1.0
Accuracy:  0.795


Answer to question:
Some synonyms can be interpreted as more negative (or positive) than the original word, which can make the documents look more positive or negative than they are. 

OUTPUT
```
Most Informative Features
         contains(owing) = True              pos : neg    =      9.7 : 1.0
  contains(undischarged) = True              pos : neg    =      9.7 : 1.0
           synset(owing) = True              pos : neg    =      9.7 : 1.0
    synset(undischarged) = True              pos : neg    =      9.7 : 1.0
    contains(squandered) = True              neg : pos    =      6.4 : 1.0
      synset(squandered) = True              neg : pos    =      6.4 : 1.0
      contains(farcical) = True              neg : pos    =      5.8 : 1.0
        synset(farcical) = True              neg : pos    =      5.8 : 1.0
     contains(atrophied) = True              neg : pos    =      5.6 : 1.0
       synset(atrophied) = True              neg : pos    =      5.6 : 1.0
Accuracy:  0.795
```


## Exercise 4 -- Experimentation
This exercise is largely open to experiment with and testing your skills thus far!
Large websites are an ideal place to look for large corpora of natural language. In this exercise, you're free to implement what you've learned on real-world data, mined from youtube (see `youtube_data`). Reuse classes defined earlier on in the exercise if you want.

The only requirement here is to **use a classifier not previously used in the exercise**

In [45]:
# Find out who made a video

youtubeData = pd.read_csv(
    'youtube_data/.ipynb_checkpoints/videos-checkpoint.csv',
    usecols=["channel_title", "title", "tags"],
    encoding="latin-1",
    nrows=1000
)

def cleanTag(tag):
    tag = tag.replace('|', " ")
    tag = tag.replace('"', "")
    return tag

def cleanTitle(title):
    clean = TextCleaner(title).clean()
    return clean

def generateFeatures(tag, title):
    tag = cleanTag(tag)
    title = cleanTitle(title)
    return {"tags": tag, "title": title, "numberOfTags": len(tag.split(" ")), "lengthOfTitle": len(title)}

featuresets = [(generateFeatures(row.tags, row.title), row.channel_title) for i, row in youtubeData.iterrows()]

trainSet, testSet = train_test_split(featuresets, test_size=0.3)

classifier = nltk.MaxentClassifier.train(trainSet)

classifier.show_most_informative_features()

print(nltk.classify.accuracy(classifier, testSet))

youtubeData.head()



  ==> Training (100 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -5.74300        0.001
             2          -0.77731        1.000
             3          -0.45345        1.000
             4          -0.32192        1.000
             5          -0.25005        1.000
             6          -0.20464        1.000
             7          -0.17330        1.000
             8          -0.15037        1.000
             9          -0.13284        1.000
            10          -0.11901        1.000
            11          -0.10781        1.000
            12          -0.09855        1.000
            13          -0.09078        1.000
            14          -0.08415        1.000
            15          -0.07844        1.000
            16          -0.07346        1.000
            17          -0.06908        1.000
            18          -0.06520        1.000
            19          -0.06173        1.000
 

,title,channel_title,tags
0,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,SHANtell martin
1,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,"last week tonight trump presidency|""last week ..."
2,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,"racist superman|""rudy""|""mancuso""|""king""|""bach""..."
3,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,"rhett and link|""gmm""|""good mythical morning""|""..."
4,I Dare You: GOING BALD!?,nigahiga,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""..."


OUTPUT
```
   7.301 title=='donald trump makes asean handshake photo op go awry' and label is 'Rappler'
   6.272 title=='f the prom ( 2017 ) | official trailer hd' and label is 'The Orchard Movies'
   6.272 title=='# staronfox exclusive clip for theybf.com' and label is 'YBF Chic'
   5.568 title=='sphaera - demonstrating interaction' and label is 'Jenny Hanell'
   5.465 title=='watch : sen. mitch mcconnell on tax reform' and label is 'WLKY News Louisville'
   5.462 title=='baby loves jeopardy !' and label is 'Daelric'
   5.352 title=='alicia keys - when you were gone' and label is 'Alicia Keys'
   5.261 title=='president bill clinton on dictators , democracy , & why we need immigrants more than ever' and label is 'Team Coco'
   5.185 lengthOfTitle==89 and label is 'Team Coco'
   5.019 title=='dennis smith jr. and lebron james go back and forth' and label is 'Ben Rohrbach'
0.7633333333333333

```